# Board game and car dealership review texts with ratings, sentiment labels, and aggregated review counts Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In this notebook, we will:
- load the Croissant metadata,
- inspect record sets and fields using their `@id` values,
- attempt to extract records from every available record set,
- perform lightweight exploratory data analysis,
- and create a few visualizations when compatible tabular fields are available.

All references to record sets and fields are made using their `@id` values, consistent with Croissant best practices.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`.

We keep the dataset URL in a variable so it can be reused throughout the notebook.

In [ ]:
import json
import math
from collections import Counter

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.3wpp-c1wp/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset name:", getattr(metadata, "name", "<missing>"))
print("Description:", getattr(metadata, "description", "<missing>"))
print("Identifier:", getattr(metadata, "identifier", "<missing>"))
print("Version:", getattr(metadata, "version", "<missing>"))
print("License:", getattr(metadata, "license", "<missing>"))

In [ ]:
# A compact metadata summary as a regular Python object for inspection if needed
metadata_json = metadata.to_json()
print(json.dumps({
    "name": metadata_json.get("name"),
    "description": metadata_json.get("description"),
    "recordSet_count": len(metadata_json.get("recordSet", []) or []),
    "distribution_count": len(metadata_json.get("distribution", []) or []),
    "keywords": metadata_json.get("keywords", [])[:10],
}, indent=2))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Because Croissant datasets are schema-driven, it is useful to inspect the metadata first and then use the discovered `@id` values to load records.

This dataset package may expose zero or more record sets. The code below safely inspects whatever is present in the schema.

In [ ]:
record_sets_metadata = metadata_json.get("recordSet", []) or []
print(f"Number of record sets declared in metadata: {len(record_sets_metadata)}")

overview_rows = []
for rs in record_sets_metadata:
    rs_id = rs.get("@id", "<missing_record_set_id>")
    rs_name = rs.get("name", "")
    rs_description = rs.get("description", "")
    fields = rs.get("field", []) or []
    overview_rows.append({
        "record_set_id": rs_id,
        "record_set_name": rs_name,
        "field_count": len(fields),
        "description": rs_description,
    })

overview_df = pd.DataFrame(overview_rows)
overview_df

In [ ]:
# Detailed field-level overview using `@id` values
field_rows = []

for rs in record_sets_metadata:
    rs_id = rs.get("@id", "<missing_record_set_id>")
    rs_name = rs.get("name", "")
    fields = rs.get("field", []) or []
    for field in fields:
        source_obj = field.get("source", {}) if isinstance(field.get("source", {}), dict) else {}
        field_rows.append({
            "record_set_id": rs_id,
            "record_set_name": rs_name,
            "field_id": field.get("@id", "<missing_field_id>"),
            "field_name": field.get("name", ""),
            "data_type": field.get("dataType", ""),
            "description": field.get("description", ""),
            "source_file_object_id": source_obj.get("fileObject", "") if isinstance(source_obj, dict) else "",
            "source_extract": source_obj.get("extract", {}) if isinstance(source_obj, dict) else {},
        })

fields_df = pd.DataFrame(field_rows)
fields_df.head(20)

In [ ]:
# Example iteration over one record set, using its @id if available
if record_sets_metadata:
    example_record_set_id = record_sets_metadata[0].get("@id")
    print("Example record_set @id:", example_record_set_id)
    try:
        for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
            print(x)
            if i >= 2:
                break
    except Exception as e:
        print(f"Could not iterate records for {example_record_set_id}: {e}")
else:
    print("No record sets were declared in the metadata, so there are no tabular records to iterate yet.")

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis.

We dynamically gather record set IDs from the metadata so the notebook adapts to the dataset schema without hard-coding assumptions.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.get("@id") for rs in record_sets_metadata if rs.get("@id")]
dataframes = {}
load_status_rows = []

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        load_status_rows.append({
            "record_set_id": record_set_id,
            "loaded": True,
            "n_rows": len(df),
            "n_columns": len(df.columns),
            "error": ""
        })
    except Exception as e:
        load_status_rows.append({
            "record_set_id": record_set_id,
            "loaded": False,
            "n_rows": None,
            "n_columns": None,
            "error": str(e)
        })

load_status_df = pd.DataFrame(load_status_rows)
load_status_df

In [ ]:
# Inspect the first successfully loaded record set, if one exists
loaded_record_set_ids = [rs_id for rs_id, df in dataframes.items() if isinstance(df, pd.DataFrame)]

if loaded_record_set_ids:
    first_record_set_id = loaded_record_set_ids[0]
    print("Selected record set @id:", first_record_set_id)
    print("Columns:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No record sets were successfully loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as type inspection, missing value review, filtering, normalization, grouping, and light text analysis.

Because Croissant datasets can vary, the workflow below automatically chooses a usable record set and field candidates where possible.

In [ ]:
def get_numeric_columns(df: pd.DataFrame):
    numeric_cols = []
    for col in df.columns:
        coerced = pd.to_numeric(df[col], errors="coerce")
        if coerced.notna().sum() > 0:
            numeric_cols.append(col)
    return numeric_cols

def get_text_like_columns(df: pd.DataFrame):
    text_cols = []
    for col in df.columns:
        if df[col].dtype == object:
            text_cols.append(col)
    return text_cols

def choose_group_column(df: pd.DataFrame):
    candidates = []
    for col in df.columns:
        nunique = df[col].nunique(dropna=True)
        if 1 < nunique <= min(25, max(len(df) // 2, 2)):
            candidates.append((col, nunique))
    return candidates[0][0] if candidates else None

analysis_record_set_id = None
analysis_df = None

for rs_id in loaded_record_set_ids:
    df = dataframes[rs_id].copy()
    if len(df) > 0:
        analysis_record_set_id = rs_id
        analysis_df = df
        break

print("Analysis record set @id:", analysis_record_set_id)
if analysis_df is not None:
    print("Shape:", analysis_df.shape)
    display(analysis_df.head())
else:
    print("No suitable DataFrame available for EDA.")

In [ ]:
if analysis_df is not None:
    missing_summary = pd.DataFrame({
        "column": analysis_df.columns,
        "dtype": [str(analysis_df[c].dtype) for c in analysis_df.columns],
        "missing_count": [analysis_df[c].isna().sum() for c in analysis_df.columns],
        "missing_pct": [analysis_df[c].isna().mean() * 100 for c in analysis_df.columns],
        "n_unique": [analysis_df[c].nunique(dropna=True) for c in analysis_df.columns],
    }).sort_values(["missing_pct", "n_unique"], ascending=[False, False])
    display(missing_summary)
else:
    print("Skipping missing-value summary because no analysis DataFrame is available.")

In [ ]:
# Select a numeric field for analysis dynamically
if analysis_df is not None:
    numeric_fields = get_numeric_columns(analysis_df)
    print("Candidate numeric field IDs/columns:", numeric_fields)
else:
    numeric_fields = []

if analysis_df is not None and numeric_fields:
    numeric_field = numeric_fields[0]
    analysis_df[numeric_field] = pd.to_numeric(analysis_df[numeric_field], errors="coerce")
    threshold = analysis_df[numeric_field].quantile(0.50)
    filtered_df = analysis_df[analysis_df[numeric_field] > threshold].copy()
    print(f"Using numeric field: {numeric_field}")
    print(f"Threshold (median): {threshold}")
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    std_value = filtered_df[numeric_field].std()
    if pd.notna(std_value) and std_value != 0:
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
        )
    else:
        filtered_df[f"{numeric_field}_normalized"] = 0.0

    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    group_field = choose_group_column(filtered_df)
    print("Selected group field:", group_field)
    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).to_frame(name=f"mean_{numeric_field}")
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head(10))
    else:
        print("No suitable grouping field was found.")
else:
    print("No numeric field was available for filtering and normalization.")
    filtered_df = None
    numeric_field = None
    group_field = None

In [ ]:
# Optional light text analysis when text columns are present
if analysis_df is not None:
    text_fields = get_text_like_columns(analysis_df)
    print("Candidate text field IDs/columns:", text_fields)
else:
    text_fields = []

if analysis_df is not None and text_fields:
    text_field = text_fields[0]
    text_series = analysis_df[text_field].dropna().astype(str)
    text_lengths = text_series.str.len()
    text_word_counts = text_series.str.split().str.len()

    text_stats = pd.DataFrame({
        "metric": ["non_null_texts", "avg_char_length", "median_char_length", "avg_word_count", "median_word_count"],
        "value": [
            int(text_series.shape[0]),
            float(text_lengths.mean()) if len(text_lengths) else math.nan,
            float(text_lengths.median()) if len(text_lengths) else math.nan,
            float(text_word_counts.mean()) if len(text_word_counts) else math.nan,
            float(text_word_counts.median()) if len(text_word_counts) else math.nan,
        ]
    })
    print(f"Text analysis for field: {text_field}")
    display(text_stats)

    tokens = []
    for text in text_series.head(1000):
        tokens.extend([tok.lower() for tok in str(text).split() if len(tok) > 3])
    common_terms = pd.DataFrame(Counter(tokens).most_common(20), columns=["term", "count"])
    display(common_terms)
else:
    print("No text-like field was available for text analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The plotting logic below runs only when suitable columns are available.

In [ ]:
if analysis_df is not None and numeric_fields:
    plot_numeric_field = numeric_field if numeric_field is not None else numeric_fields[0]
    plot_df = analysis_df.copy()
    plot_df[plot_numeric_field] = pd.to_numeric(plot_df[plot_numeric_field], errors="coerce")
    plot_df = plot_df.dropna(subset=[plot_numeric_field])

    if len(plot_df) > 0:
        fig, ax = plt.subplots(1, 2, figsize=(14, 5))
        sns.histplot(plot_df[plot_numeric_field], kde=True, ax=ax[0], color="steelblue")
        ax[0].set_title(f"Distribution of {plot_numeric_field}")
        ax[0].set_xlabel(plot_numeric_field)

        sns.boxplot(x=plot_df[plot_numeric_field], ax=ax[1], color="orange")
        ax[1].set_title(f"Boxplot of {plot_numeric_field}")
        ax[1].set_xlabel(plot_numeric_field)

        plt.tight_layout()
        plt.show()
    else:
        print("Numeric plotting skipped because the selected numeric field has no non-null values.")
else:
    print("Numeric plotting skipped because no numeric field is available.")

In [ ]:
if analysis_df is not None and group_field is not None and numeric_field is not None and group_field in analysis_df.columns:
    group_plot_df = analysis_df.copy()
    group_plot_df[numeric_field] = pd.to_numeric(group_plot_df[numeric_field], errors="coerce")
    group_plot_df = group_plot_df.dropna(subset=[numeric_field, group_field])

    if len(group_plot_df) > 0:
        grouped_plot = (
            group_plot_df.groupby(group_field)[numeric_field]
            .mean()
            .sort_values(ascending=False)
            .head(15)
        )

        plt.figure(figsize=(12, 5))
        sns.barplot(x=grouped_plot.index.astype(str), y=grouped_plot.values, palette="viridis")
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Grouped plotting skipped because there are no valid rows for the selected fields.")
else:
    print("Grouped plotting skipped because suitable grouping or numeric fields were not found.")

In [ ]:
if analysis_df is not None and text_fields:
    plot_text_field = text_fields[0]
    text_plot_series = analysis_df[plot_text_field].dropna().astype(str)
    if len(text_plot_series) > 0:
        text_length_df = pd.DataFrame({
            "char_length": text_plot_series.str.len(),
            "word_count": text_plot_series.str.split().str.len()
        })

        fig, ax = plt.subplots(1, 2, figsize=(14, 5))
        sns.histplot(text_length_df["char_length"], bins=30, ax=ax[0], color="purple")
        ax[0].set_title(f"Character length distribution for {plot_text_field}")
        ax[0].set_xlabel("Character length")

        sns.histplot(text_length_df["word_count"], bins=30, ax=ax[1], color="green")
        ax[1].set_title(f"Word count distribution for {plot_text_field}")
        ax[1].set_xlabel("Word count")

        plt.tight_layout()
        plt.show()
    else:
        print("Text plotting skipped because the selected text field has no non-null values.")
else:
    print("Text plotting skipped because no text-like field is available.")

## 6. Conclusion
This notebook demonstrated a reusable Croissant-based workflow for:
- loading dataset metadata with `mlcroissant`,
- identifying record sets and fields by `@id`,
- extracting records into pandas DataFrames,
- performing basic filtering, normalization, grouping, and text inspection,
- and creating visual summaries when suitable fields are available.

For this particular dataset package, the metadata indicates a review dataset spanning BoardGameGeek and TrustPilot, with review-level and aggregate-count information described in the package metadata. If record sets are absent or not directly exposed in the schema, the metadata inspection cells still provide a useful starting point for understanding the dataset structure and validating what is available through the Croissant endpoint.